# NeuroScan AI: Preprocesamiento y Pipeline tf.data
## Módulo 3: Databricks e IA Aplicada - Proyecto Final

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EduardCY/m3-proyecto-final-databricks-ia/blob/main/notebooks/02_preprocessing.ipynb)

Este notebook implementa la **Etapa 2**:
1. Construcción de canales estandarizados `(224, 224, 3)`.
2. Normalización de pesos ImageNet (BGR mean centering).
3. Data Augmentation clínicamente seguro para neuroimagen.
4. Optimización de pipeline con `prefetch(AUTOTUNE)` y verificación de lotes.


In [ ]:
# 0. Inicialización Automática de Entorno (Google Colab / Local)
import os
import sys

# Detección y preparación autónoma para Google Colab
if 'google.colab' in sys.modules:
    # 1. Clonar o actualizar el repositorio público en /content
    if not os.path.exists('/content/m3-proyecto-final-databricks-ia'):
        print("🚀 Clonando repositorio NeuroScan AI en Google Colab...")
        !git clone https://github.com/EduardCY/m3-proyecto-final-databricks-ia.git /content/m3-proyecto-final-databricks-ia
    else:
        print("🔄 Actualizando repositorio con los últimos cambios de GitHub...")
        !git -C /content/m3-proyecto-final-databricks-ia pull origin main
    
    # 2. Posicionarse en la carpeta raíz del proyecto
    %cd /content/m3-proyecto-final-databricks-ia
    
    # 3. Instalar dependencias requeridas para Colab (MLflow, etc.)
    !pip install -q -r requirements-colab.txt

    # 4. Limpiar módulos en caché para recargar cambios de código
    for mod in list(sys.modules.keys()):
        if mod.startswith('src') or mod.startswith('scripts'):
            del sys.modules[mod]

# 5. Asegurar que la raíz del proyecto está en el PYTHONPATH
PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"✅ Entorno preparado en: {PROJECT_ROOT}")


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from src.config import BATCH_SIZE, CLASSES, IMG_CHANNELS, IMG_SIZE, NUM_CLASSES, SPLIT_MANIFEST_PATH
from src.preprocessing import create_tf_dataset

print(f"TensorFlow Version: {tf.__version__}")
print(f"Dispositivos disponibles: {tf.config.list_physical_devices()}")


### 1. Cargar el Split de Datos


In [ ]:
from scripts.download_dataset import ensure_dataset_ready
from src.dataset import build_manifest, split_manifest
from src.config import DATA_DIR, SPLIT_MANIFEST_PATH

ensure_dataset_ready(DATA_DIR)
if not SPLIT_MANIFEST_PATH.exists():
    print("Partición no encontrada en disco. Generando split estratificado...")
    manifest_df = build_manifest(data_dir=DATA_DIR, use_cache=False)
    df_split = split_manifest(manifest_df, output_path=SPLIT_MANIFEST_PATH)
else:
    df_split = pd.read_csv(SPLIT_MANIFEST_PATH)

print("Resumen de splits:")
print(df_split.groupby(["split", "label_name"]).size().unstack())


### 2. Construir Pipelines tf.data (Train, Val, Test)


In [ ]:
train_ds = create_tf_dataset(df_split, split_name="train", batch_size=BATCH_SIZE, model_type="resnet50")
val_ds = create_tf_dataset(df_split, split_name="val", batch_size=BATCH_SIZE, model_type="resnet50")
test_ds = create_tf_dataset(df_split, split_name="test", batch_size=BATCH_SIZE, model_type="resnet50")

print(f"Pipeline construido: Lotes de tamaño {BATCH_SIZE} listos para GPU/CPU.")


### 3. Inspección Visual de un Lote


In [ ]:
for images, labels in train_ds.take(1):
    print(f"Tensor de imágenes: {images.shape}, tipo: {images.dtype}")
    print(f"Tensor de etiquetas: {labels.shape}, tipo: {labels.dtype}")
    
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for i, ax in enumerate(axes.flat):
        img_np = images[i].numpy()
        # Deshacer centrado de media para visualización legible
        img_disp = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-6)
        label_idx = np.argmax(labels[i].numpy())
        ax.imshow(img_disp)
        ax.set_title(f"Clase: {CLASSES[label_idx]}")
        ax.axis("off")
    plt.suptitle("Muestras de Resonancias Preprocesadas con Data Augmentation")
    plt.tight_layout()
    plt.show()
